# 08 — Business Health Score

**AI Financial Intelligence Platform — ML Module**

---

## Purpose

Compute a composite AI-generated **Business Health Score** (0–100) per company and branch.

### Score Components
| Dimension | Weight | Source |
|-----------|--------|--------|
| Revenue Growth | 25% | sales |
| Profit Margin | 25% | sales + expenses |
| Expense Control | 20% | expenses |
| Inventory Turnover | 15% | inventory + stock_movements |
| Payment Collection | 15% | invoices + payments |

### Sections
1. Load all relevant tables
2. Compute individual KPIs
3. Normalize KPIs to 0–100
4. Compute weighted composite score
5. Visualize score per company/branch over time
6. Supervised Model (Future)
7. Save scoring function


In [ ]:
import os
import json
import pickle
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

DATA_DIR    = '../datasets/synthetic/'
MODELS_DIR  = '../models/'
REPORTS_DIR = '../reports/'
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

## 1. Load All Relevant Tables


In [ ]:
# Load data
sales           = pd.read_csv(DATA_DIR + 'sales.csv')
sale_items      = pd.read_csv(DATA_DIR + 'sale_items.csv')
expenses        = pd.read_csv(DATA_DIR + 'expenses.csv')
invoices        = pd.read_csv(DATA_DIR + 'invoices.csv')
payments        = pd.read_csv(DATA_DIR + 'payments.csv')
inventory       = pd.read_csv(DATA_DIR + 'inventory.csv')
stock_movements = pd.read_csv(DATA_DIR + 'stock_movements.csv')
products        = pd.read_csv(DATA_DIR + 'products.csv')
companies       = pd.read_csv(DATA_DIR + 'companies.csv')

# Parse dates
sales['sale_date']            = pd.to_datetime(sales['sale_date'],            errors='coerce')
expenses['expense_date']      = pd.to_datetime(expenses['expense_date'],      errors='coerce')
invoices['invoice_date']      = pd.to_datetime(invoices['invoice_date'],      errors='coerce')
invoices['due_date']          = pd.to_datetime(invoices['due_date'],          errors='coerce')
payments['payment_date']      = pd.to_datetime(payments['payment_date'],      errors='coerce')
stock_movements['moved_at']   = pd.to_datetime(stock_movements['moved_at'],   errors='coerce')

print('All tables loaded successfully.')
for name, df in [('sales', sales), ('expenses', expenses), ('invoices', invoices),
                  ('payments', payments), ('inventory', inventory),
                  ('stock_movements', stock_movements)]:
    print(f'  {name:<20}: {df.shape}')

## 2. Compute Individual KPIs


In [ ]:
# ── KPI 1: Monthly Revenue Growth Rate ────────────────────────────────────────
monthly_rev = (
    sales.dropna(subset=['sale_date'])
    .groupby(['company_id', pd.Grouper(key='sale_date', freq='ME')])
    .agg(revenue=('net_amount', 'sum'))
    .reset_index()
    .rename(columns={'sale_date': 'month'})
    .sort_values(['company_id', 'month'])
)
monthly_rev['revenue_lag1']   = monthly_rev.groupby('company_id')['revenue'].shift(1)
monthly_rev['revenue_growth'] = (
    (monthly_rev['revenue'] - monthly_rev['revenue_lag1']) /
    monthly_rev['revenue_lag1'].replace(0, np.nan)
).clip(-1, 5)   # clip extreme values

# ── KPI 2: Profit Margin ──────────────────────────────────────────────────────
items_cost = (
    sale_items
    .merge(products[['id', 'cost_price']], left_on='product_id', right_on='id')
    .merge(sales[['id', 'sale_date', 'company_id']], left_on='sale_id', right_on='id',
           suffixes=('_item', '_sale'))
)
items_cost['cogs'] = items_cost['quantity'] * items_cost['cost_price']

monthly_cogs = (
    items_cost
    .groupby(['company_id', pd.Grouper(key='sale_date', freq='ME')])
    .agg(cogs=('cogs', 'sum'))
    .reset_index()
    .rename(columns={'sale_date': 'month'})
)

monthly_exp = (
    expenses.dropna(subset=['expense_date'])
    .groupby(['company_id', pd.Grouper(key='expense_date', freq='ME')])
    .agg(total_exp=('amount', 'sum'))
    .reset_index()
    .rename(columns={'expense_date': 'month'})
)

profit_df = (
    monthly_rev
    .merge(monthly_cogs, on=['company_id', 'month'], how='left')
    .merge(monthly_exp,  on=['company_id', 'month'], how='left')
)
profit_df['cogs']       = profit_df['cogs'].fillna(0)
profit_df['total_exp']  = profit_df['total_exp'].fillna(0)
profit_df['gross_profit'] = profit_df['revenue'] - profit_df['cogs']
profit_df['net_profit']   = profit_df['gross_profit'] - profit_df['total_exp']
profit_df['profit_margin'] = (
    profit_df['net_profit'] / profit_df['revenue'].replace(0, np.nan)
).clip(-1, 1)

# ── KPI 3: Expense Control (expense ratio = expenses / revenue) ───────────────
profit_df['expense_ratio'] = (
    profit_df['total_exp'] / profit_df['revenue'].replace(0, np.nan)
).clip(0, 2)
# Lower expense ratio = better control → invert later during scoring

# ── KPI 4: Inventory Turnover (COGS / avg inventory value) ───────────────────
inv_value = (
    inventory
    .merge(products[['id', 'cost_price']], left_on='product_id', right_on='id')
    .assign(stock_value=lambda d: d['quantity_on_hand'] * d['cost_price'])
    .groupby('company_id')
    .agg(avg_inventory_value=('stock_value', 'mean'))
    .reset_index()
)

annual_cogs = items_cost.groupby('company_id')['cogs'].sum().reset_index(name='annual_cogs')
turnover_df = inv_value.merge(annual_cogs, on='company_id', how='left')
turnover_df['inventory_turnover'] = (
    turnover_df['annual_cogs'] / turnover_df['avg_inventory_value'].replace(0, np.nan)
).fillna(0)

# ── KPI 5: Payment Collection Rate (paid invoices / total invoices) ───────────
inv_monthly = (
    invoices.dropna(subset=['invoice_date'])
    .groupby(['company_id', pd.Grouper(key='invoice_date', freq='ME')])
    .agg(
        total_invoiced = ('total_amount', 'sum'),
        paid_invoiced  = ('total_amount', lambda x: x[invoices.loc[x.index, 'status'] == 'paid'].sum()),
    )
    .reset_index()
    .rename(columns={'invoice_date': 'month'})
)
inv_monthly['collection_rate'] = (
    inv_monthly['paid_invoiced'] / inv_monthly['total_invoiced'].replace(0, np.nan)
).fillna(0).clip(0, 1)

print('KPIs computed successfully.')
print('Revenue growth — sample:', monthly_rev['revenue_growth'].describe().round(3).to_dict())

## 3. Normalize KPIs (0–100)


In [ ]:
# ── Merge all KPIs into a single monthly company-level frame ──────────────────
kpi_df = (
    profit_df[['company_id', 'month', 'revenue_growth', 'profit_margin', 'expense_ratio']]
    .merge(inv_monthly[['company_id', 'month', 'collection_rate']],
           on=['company_id', 'month'], how='left')
)

# Add (company-level static) inventory turnover — broadcast to monthly
kpi_df = kpi_df.merge(turnover_df[['company_id', 'inventory_turnover']],
                       on='company_id', how='left')

kpi_df = kpi_df.dropna(subset=['revenue_growth']).copy()

def minmax_to_100(series: pd.Series) -> pd.Series:
    """Scale a series to [0, 100] using global min-max."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(50.0, index=series.index)
    return ((series - mn) / (mx - mn)) * 100

# ── Normalize each KPI ────────────────────────────────────────────────────────
# Higher revenue_growth  → better
kpi_df['score_revenue_growth']    = minmax_to_100(kpi_df['revenue_growth'])

# Higher profit_margin   → better
kpi_df['score_profit_margin']     = minmax_to_100(kpi_df['profit_margin'])

# Lower expense_ratio    → better (invert)
kpi_df['score_expense_control']   = 100 - minmax_to_100(kpi_df['expense_ratio'])

# Higher inventory_turnover → better
kpi_df['score_inventory_turnover']= minmax_to_100(kpi_df['inventory_turnover'])

# Higher collection_rate → better
kpi_df['score_payment_collection']= minmax_to_100(
    kpi_df['collection_rate'].fillna(kpi_df['collection_rate'].median())
)

print('Normalized KPI score ranges:')
score_cols = [c for c in kpi_df.columns if c.startswith('score_')]
print(kpi_df[score_cols].describe().round(1).to_string())

## 4. Composite Health Score


In [ ]:
WEIGHTS = {
    'revenue_growth':    0.25,
    'profit_margin':     0.25,
    'expense_control':   0.20,
    'inventory_turnover':0.15,
    'payment_collection':0.15,
}

# Compute weighted sum to produce health_score (0–100)
kpi_df['health_score'] = (
    kpi_df['score_revenue_growth']     * WEIGHTS['revenue_growth'] +
    kpi_df['score_profit_margin']      * WEIGHTS['profit_margin'] +
    kpi_df['score_expense_control']    * WEIGHTS['expense_control'] +
    kpi_df['score_inventory_turnover'] * WEIGHTS['inventory_turnover'] +
    kpi_df['score_payment_collection'] * WEIGHTS['payment_collection']
).round(2)

# Health labels
def health_label(score):
    if score >= 75:   return 'Excellent'
    elif score >= 55: return 'Good'
    elif score >= 35: return 'Fair'
    else:             return 'At Risk'

kpi_df['health_label'] = kpi_df['health_score'].apply(health_label)

# Latest snapshot per company
latest_score = (
    kpi_df
    .merge(companies[['id', 'name']], left_on='company_id', right_on='id', how='left')
    .sort_values('month')
    .groupby('company_id')
    .last()
    .reset_index()
    [['company_id', 'name', 'health_score', 'health_label'] + score_cols]
)
print('=== Latest Business Health Scores ===')
print(latest_score[['name', 'health_score', 'health_label']].to_string(index=False))

## 5. Visualize Health Score


In [ ]:
# ── Line chart of health score over time per company ─────────────────────────
kpi_plot = kpi_df.merge(companies[['id', 'name']], left_on='company_id', right_on='id', how='left')

fig, ax = plt.subplots(figsize=(14, 5))
for cname, grp in kpi_plot.groupby('name'):
    ax.plot(grp['month'], grp['health_score'], lw=2, marker='o', markersize=3, label=cname)

# Health zone bands
ax.axhspan(75, 100, alpha=0.05, color='green',  label='Excellent (75–100)')
ax.axhspan(55,  75, alpha=0.05, color='blue',   label='Good (55–75)')
ax.axhspan(35,  55, alpha=0.05, color='orange', label='Fair (35–55)')
ax.axhspan(0,   35, alpha=0.05, color='red',    label='At Risk (0–35)')

ax.set_title('Business Health Score Over Time — Per Company', fontweight='bold', fontsize=13)
ax.set_ylabel('Health Score (0–100)')
ax.set_ylim(0, 100)
ax.legend(title='Company', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# ── Spider / Radar chart for latest scores per company ────────────────────────
dimensions = list(WEIGHTS.keys())
score_map  = {
    'revenue_growth':     'score_revenue_growth',
    'profit_margin':      'score_profit_margin',
    'expense_control':    'score_expense_control',
    'inventory_turnover': 'score_inventory_turnover',
    'payment_collection': 'score_payment_collection',
}

fig = go.Figure()
for _, row in latest_score.iterrows():
    values = [row[score_map[d]] for d in dimensions]
    values += [values[0]]  # close the polygon
    theta  = dimensions + [dimensions[0]]
    fig.add_trace(go.Scatterpolar(
        r=values, theta=theta,
        fill='toself', name=row['name'],
        opacity=0.7,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Business Health KPI Radar — Latest Period',
    showlegend=True,
    width=700, height=500,
)
fig.show()

# ── Heatmap: health score over months × companies ────────────────────────────
pivot_heat = kpi_plot.pivot_table(
    index='name', columns='month', values='health_score', aggfunc='mean'
)

fig2, ax2 = plt.subplots(figsize=(16, max(3, len(pivot_heat) * 0.6)))
sns.heatmap(pivot_heat, cmap='RdYlGn', vmin=0, vmax=100,
            linewidths=0.3, cbar_kws={'label': 'Health Score'},
            fmt='.0f', annot=len(pivot_heat.columns) <= 30, ax=ax2)
ax2.set_title('Business Health Score Heatmap (Company × Month)', fontweight='bold')
ax2.set_xlabel('Month')
ax2.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()

## 6. Supervised Model (Future)


In [ ]:
# NOTE: This section is a framework placeholder for when labeled health data becomes available.
# When labeled data is available, train a classification/regression model as follows:

# Example schema expected:
#   labeled_df: DataFrame with columns [company_id, month, health_score, label]
#   where 'label' is manually assigned by business analysts (e.g., 0=unhealthy, 1=healthy)

# Example training pipeline (commented out until labels are available):
# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report
#
# X = kpi_df[score_cols].fillna(0)
# y = (kpi_df['health_score'] >= 55).astype(int)  # proxy label
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# clf = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
# clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)
# print(classification_report(y_test, y_pred))

print('Supervised model section: placeholder ready.')
print('Provide labeled health data to activate training.')
print('Current score distribution:')
print(kpi_df['health_label'].value_counts().to_string())

## 7. Save Scoring Function


In [ ]:
# ── Save the scoring pipeline as a pickle ─────────────────────────────────────
scoring_pipeline = {
    'weights': WEIGHTS,
    'score_cols': score_cols,
    'kpi_ranges': {
        'revenue_growth':     (float(kpi_df['revenue_growth'].min()),     float(kpi_df['revenue_growth'].max())),
        'profit_margin':      (float(kpi_df['profit_margin'].min()),      float(kpi_df['profit_margin'].max())),
        'expense_ratio':      (float(kpi_df['expense_ratio'].min()),      float(kpi_df['expense_ratio'].max())),
        'inventory_turnover': (float(kpi_df['inventory_turnover'].min()), float(kpi_df['inventory_turnover'].max())),
        'collection_rate':    (float(kpi_df['collection_rate'].fillna(0).min()),
                               float(kpi_df['collection_rate'].fillna(0).max())),
    }
}

pipeline_path = os.path.join(MODELS_DIR, 'health_score_pipeline.pkl')
with open(pipeline_path, 'wb') as f:
    pickle.dump(scoring_pipeline, f)
print(f'Scoring pipeline saved → {pipeline_path}')

# Save JSON version of pipeline config for API integration
with open(os.path.join(MODELS_DIR, 'health_score_config.json'), 'w') as f:
    json.dump(scoring_pipeline, f, indent=2, default=float)
print('Scoring config (JSON) saved.')

# Save full KPI + score DataFrame for reporting
kpi_df.to_csv(os.path.join(REPORTS_DIR, 'business_health_scores.csv'), index=False)
latest_score.to_csv(os.path.join(REPORTS_DIR, 'latest_health_scores.csv'), index=False)
print('Health score reports saved to reports/ directory.')